# 1 Experiment Setup

```
├── Dockerfile
├── README.md
├── data
│   ├── covariate_lag_selection_results.csv
│   ├── dataset_info.csv
│   ├── dim_length.csv
│   ├── forecasts
│   │   ├── data.csv
│   │   ├── meta.csv
│   ├── metrics.csv
│   ├── plots
│   ├── prepared_datasets
│   │   ├── air_quality.csv
│   │   ├── amiata.csv
│   │   ├── arno.csv
│   │   ├── auser.csv
│   │   ├── bike_sharing.csv
│   │   ├── bilancino.csv
│   │   ├── bitcoin.csv
│   │   ├── covid.csv
│   │   ├── covid_mobility.csv
│   │   ├── demo.csv
│   │   ├── doganella.csv
│   │   ├── electricity.csv
│   │   ├── electricity_price.csv
│   │   ├── energy.csv
│   │   ├── etth.csv
│   │   ├── household_power_consumption.csv
│   │   ├── housing.csv
│   │   ├── illness.csv
│   │   ├── luco.csv
│   │   ├── lupa.csv
│   │   ├── m5_id.csv
│   │   ├── m5_other.csv
│   │   ├── madrid19.csv
│   │   ├── madrid22.csv
│   │   ├── petrignano.csv
│   │   ├── rideshare.csv
│   │   ├── stackoverflow.csv
│   │   ├── weather.csv
│   │   └── wind.csv
│   ├── sample_info.csv
│   └── timeseries_info.csv
├── logs
│   ├── evaluation.log
│   └── forecast.log
├── notebooks
├── requirements.in
├── requirements.txt
├── run.sh
├── src
│   │   ├── cov_selection.py
│   │   ├── data_loader.py
│   │   └── dataset.py
│   ├── evaluation
│   │   ├── evaluation.py
│   │   └── metrics.py
│   ├── forecast.py
│   ├── helper_functions.py
│   ├── main.py
│   └── models
│       ├── cpu_models
│       │   ├── baseline_models.py
│       │   └── other_models.py
│       ├── gpu_models
│       │   ├── progai.py
│       │   └── timegpt.py
│       └── schemas.py
└── tests
```

## 1.2 Data

### 1.2.0 Dataset

Loads and provides the preprocessed datasets as dataframe with covariate information and granularity

### 1.2.1 Samples
A sample represents following combination:
- dataset
- ts_name (unique time series identifier)
- context window
- forecast horizon

holds additional information for:
- target names
- ts_id

### 1.2.2 Data loader
Iterates over given number of samples and prepares model inpud based on covariate mode. Following covariates modes are supported:
- no covariates
- all covariates given in the dataset
- only past covariates
- only future covariates
- random permutation of all covariates (noise)
- no covariates from the dataset but time covariates added based on granularity (e.g. day of week)
- no covariates from the dataset but lagged versions of the targets
- selection of covariates
- selection of covariates with best lag

### 1.2.3 Covariate Selection

Trained LightGBM model for each of the covariates of an dataset with a range of lags. For each covariate lag combinations accuracy (mase) is calculated on test set. Based on accuracy covariates and corresponding lags are ranked.

## 1.3 Models

### 1.3.1 CPU

#### 1.3.1.1 Baseline Models

Auto Models of ARIMAx, ETS and Theta from statsforecast (Nixtla)

#### 1.3.1.2 Other Models

- **LightGBM:** Integrated via autogluon library
- **Chronos-Bolt:** Integrated via autogluon library with support of external regressors

### 1.3.2 GPU

#### 1.3.2.1 ProgAI

Calls API on companys GPU Cluster to get forecasts from followning models:
- DL Models: TiDE, NBEATSx and TFT
- MOIRAI (base model and moe)
- TimesFM (optional use of external regressor)
- TTM (support of finetuning)

#### 1.3.2.1 TimeGPT

Calls NIxtlas TimeGPT Client

## 1.4 Forecast

- Holds differen model configurations
- Takes config of model and mode combinations and a dataloader instance
- Iterates over configurations and samples in the dataloader and performs forecasts
- Handles batch wise storing of the forecasts and logging for error tracing

## 1.5 Evaluation
- Checks available forecasts
- Calculates different metrics over all unique forecasts
- Aggregates metrics over models and modes for further processing
- Provides different plots:
    - Barplot with errorbar for mean/median results
    - Barplot of friedman ranks and corresponding heatmap of nemenyi post-hoc test
    - Timesereis plot with forecasts
    - Barplot of timing for comparing models
    - Scatterplot of timing vs. time series length and dimension
- Get top/bottom n timeseries

## 1.6 Main

Handles Forecasting and Evaluation tasks by multiprocessing

## 2 Progai API

- FastAPI with three endpoints:
    - GET model: give detailes on the loaded model configuration
    - POST model: updates/changes the loaded model based on provided model path and configuration
    - POST forecast: Input check and preprocessing, model inference, output processing
- All requests were processed sequentially